<a href="https://colab.research.google.com/github/neeyatlotlikar/me_fy_project/blob/dev/ADF_ReRanking_MIND.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prerequisites and Dataset

In [ ]:
!pip install implicit pandas numpy scipy gensim umap-learn hdbscan -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.4 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/ME_PROJECT/Implementation

/content/drive/MyDrive/ME_PROJECT/Implementation


In [ ]:
!mkdir -p data
%cd data

/content/drive/MyDrive/ME_PROJECT/Implementation/data


I manually downloaded the dataset and uploaded the zip file to drive since it was not possible to directly download to google colab.

In [ ]:
# Unzip
!unzip MINDsmall_train.zip
!unzip MINDsmall_dev.zip

%cd ..

Archive:  MINDsmall_train.zip
   creating: MINDsmall_train/
  inflating: MINDsmall_train/behaviors.tsv  
  inflating: MINDsmall_train/news.tsv  
  inflating: MINDsmall_train/entity_embedding.vec  
  inflating: MINDsmall_train/relation_embedding.vec  
Archive:  MINDsmall_dev.zip
   creating: MINDsmall_dev/
  inflating: MINDsmall_dev/behaviors.tsv  
  inflating: MINDsmall_dev/news.tsv  
  inflating: MINDsmall_dev/entity_embedding.vec  
  inflating: MINDsmall_dev/relation_embedding.vec  
/content/drive/MyDrive/ME_PROJECT/Implementation


In [ ]:
!ls -lh data/

total 1.3G
-rw------- 1 root root  99M Feb 23 07:38 MINDlarge_dev.zip
-rw------- 1 root root 577M Feb 23 07:40 MINDlarge_test.zip
-rw------- 1 root root 506M Feb 23 07:39 MINDlarge_train.zip
drwx------ 2 root root 4.0K Aug 11  2024 MINDsmall_dev
-rw------- 1 root root  30M Feb 23 07:40 MINDsmall_dev.zip
drwx------ 2 root root 4.0K Aug 11  2024 MINDsmall_train
-rw------- 1 root root  51M Feb 23 07:40 MINDsmall_train.zip


In [ ]:
!ls -lh data/MINDsmall_dev

total 95M
-rw------- 1 root root   41M Aug 11  2024 behaviors.tsv
-rw------- 1 root root   21M Aug 11  2024 entity_embedding.vec
-rw------- 1 root root   32M Aug 11  2024 news.tsv
-rw------- 1 root root 1021K Aug 11  2024 relation_embedding.vec


In [ ]:
!ls -lh data/MINDsmall_train

total 153M
-rw------- 1 root root   88M Aug 11  2024 behaviors.tsv
-rw------- 1 root root   25M Aug 11  2024 entity_embedding.vec
-rw------- 1 root root   40M Aug 11  2024 news.tsv
-rw------- 1 root root 1021K Aug 11  2024 relation_embedding.vec


In [ ]:
!head -n 3 data/MINDsmall_train/news.tsv

N55528	lifestyle	lifestyleroyals	The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By	Shop the notebooks, jackets, and more that the royals can't live without.	https://assets.msn.com/labs/mind/AAGH0ET.html	[{"Label": "Prince Philip, Duke of Edinburgh", "Type": "P", "WikidataId": "Q80976", "Confidence": 1.0, "OccurrenceOffsets": [48], "SurfaceForms": ["Prince Philip"]}, {"Label": "Charles, Prince of Wales", "Type": "P", "WikidataId": "Q43274", "Confidence": 1.0, "OccurrenceOffsets": [28], "SurfaceForms": ["Prince Charles"]}, {"Label": "Elizabeth II", "Type": "P", "WikidataId": "Q9682", "Confidence": 0.97, "OccurrenceOffsets": [11], "SurfaceForms": ["Queen Elizabeth"]}]	[]
N19639	health	weightloss	50 Worst Habits For Belly Fat	These seemingly harmless habits are holding you back and keeping you from shedding that unwanted belly fat for good.	https://assets.msn.com/labs/mind/AAB19MK.html	[{"Label": "Adipose tissue", "Type": "C", "WikidataId": "Q193583", "Confidence": 1.0

In [ ]:
!head -n 3 data/MINDsmall_train/behaviors.tsv

1	U13740	11/11/2019 9:05:58 AM	N55189 N42782 N34694 N45794 N18445 N63302 N10414 N19347 N31801	N55689-1 N35729-0
2	U91836	11/12/2019 6:11:30 PM	N31739 N6072 N63045 N23979 N35656 N43353 N8129 N1569 N17686 N13008 N21623 N6233 N14340 N48031 N62285 N44383 N23061 N16290 N6244 N45099 N58715 N59049 N7023 N50528 N42704 N46082 N8275 N15710 N59026 N8429 N30867 N56514 N19709 N31402 N31741 N54889 N9798 N62612 N2663 N16617 N6087 N13231 N63317 N61388 N59359 N51163 N30698 N34567 N54225 N32852 N55833 N64467 N3142 N13912 N29802 N44462 N29948 N4486 N5398 N14761 N47020 N65112 N31699 N37159 N61101 N14761 N3433 N10438 N61355 N21164 N22976 N2511 N48390 N58224 N48742 N35458 N24611 N37509 N21773 N41011 N19041 N25785	N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N22407-0 N14592-0 N17059-1 N33677-0 N7821-0 N6890-0
3	U73700	11/14/2019 7:01:48 AM	N10732 N25792 N7563 N21087 N41087 N5445 N60384 N46616 N52500 N33164 N47289 N24233 N62058 N26378 N49475 N18870	N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N59685-0 N23814-

In [ ]:
!file data/MINDsmall_train/entity_embedding.vec

data/MINDsmall_train/entity_embedding.vec: ASCII text, with very long lines (970)


In [ ]:
!head -n 5 data/MINDsmall_train/entity_embedding.vec

Q41	-0.063388	-0.181451	0.057501	-0.091254	-0.076217	-0.052525	0.050500	-0.224871	-0.018145	0.030722	0.064276	0.073063	0.039489	0.159404	-0.128784	0.016325	0.026797	0.137090	0.001849	-0.059103	0.012091	0.045418	0.000591	0.211337	-0.034093	-0.074582	0.014004	-0.099355	0.170144	0.109376	-0.014797	0.071172	0.080375	0.045563	-0.046462	0.070108	0.015413	-0.020874	-0.170324	-0.001130	0.059810	0.054342	0.027358	-0.028995	-0.224508	0.066281	-0.200006	0.018186	0.082396	0.167178	-0.136239	0.055134	-0.080195	-0.001460	0.031078	-0.017084	-0.091176	-0.036916	0.124642	-0.098185	-0.054836	0.152483	-0.053712	0.092816	-0.112044	-0.072247	-0.114896	-0.036541	-0.186339	-0.160610	0.037342	-0.133474	0.110080	0.070678	-0.005586	-0.046667	-0.072010	0.086424	0.026165	0.030561	0.077888	-0.117226	0.211597	0.112512	0.079999	-0.083398	-0.121117	0.071751	-0.017654	-0.134979	-0.051949	0.001861	0.124535	-0.151043	-0.263698	-0.103607	0.020007	-0.101157	-0.091567	0.035234	
Q1860	0.060958	0.069934	0.015832	0.079471	-0.

In [ ]:
df = pd.read_csv('data/MINDsmall_train/behaviors.tsv', sep='\t', index_col=0, nrows=25, header=None)
df.head()

,1,2,3,4
0,,,,
1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [ ]:
import pandas as pd

# Peek at files
for split in ['data/MINDsmall_dev', 'data/MINDsmall_train']:
  print(f"{split} News file shape:", pd.read_csv(f'{split}/news.tsv', sep='\t', nrows=3).shape)
  print("Interactions files:")
  df = pd.read_csv(f'{split}/behaviors.tsv', sep='\t', nrows=1000, header=None)
  print(f"  behaviors.tsv: {len(df)} rows, {df[1].nunique()} users, {df[2].nunique()} click history, {df[3].nunique()} impressions")
  for f in ['entity_embedding.vec', 'relation_embedding.vec']:
    df = pd.read_csv(f'{split}/{f}', sep='\t', nrows=1000, header=None)
    print(f"  {f}: {df.shape} shape, {df.iloc[:5, 0].tolist()} sample ids, {df.shape[1] - 1} vector dimensions")
  print()


data/MINDsmall_dev News file shape: (3, 8)
Interactions files:
  behaviors.tsv: 1000 rows, 996 users, 994 click history, 961 impressions
  entity_embedding.vec: (1000, 102) shape, ['Q34433', 'Q41', 'Q56037', 'Q1860', 'Q39631'] sample ids, 101 vector dimensions
  relation_embedding.vec: (1000, 102) shape, ['P31', 'P21', 'P106', 'P735', 'P108'] sample ids, 101 vector dimensions

data/MINDsmall_train News file shape: (3, 8)
Interactions files:
  behaviors.tsv: 1000 rows, 988 users, 1000 click history, 974 impressions
  entity_embedding.vec: (1000, 102) shape, ['Q41', 'Q1860', 'Q39631', 'Q30', 'Q60'] sample ids, 101 vector dimensions
  relation_embedding.vec: (1000, 102) shape, ['P31', 'P21', 'P106', 'P735', 'P108'] sample ids, 101 vector dimensions



In [ ]:
import implicit
print("Implicit version:", implicit.__version__)
help(implicit.bpr.BayesianPersonalizedRanking)

Implicit version: 0.7.2
Help on function BayesianPersonalizedRanking in module implicit.bpr:

BayesianPersonalizedRanking(factors=100, learning_rate=0.01, regularization=0.01, dtype=<class 'numpy.float32'>, iterations=100, use_gpu=False, num_threads=0, verify_negative_samples=True, random_state=None)
    Bayesian Personalized Ranking

    A recommender model that learns  a matrix factorization embedding based off minimizing the
    pairwise ranking loss described in the paper `BPR: Bayesian Personalized Ranking from Implicit
    Feedback <https://arxiv.org/pdf/1205.2618.pdf>`_.

    This factory function returns either the cpu implementation from implicit.cpu.bpr or
    the gpu implementation from implicit.gpu.bpr depending on the value of the use_gpu flag.

    Parameters
    ----------
    factors : int, optional
        The number of latent factors to compute
    learning_rate : float, optional
        The learning rate to apply for SGD updates during training
    regularization : f

# Load & Explore MIND Data Structure

In [4]:
import pandas as pd
import numpy as np

print("Loading MIND Data...")
behv_cols = ['impr_id', 'user_id', 'time', 'hist', 'imprs']
train_beh = pd.read_csv('data/MINDsmall_train/behaviors.tsv', sep='\t', names=behv_cols, nrows=10000)
dev_beh = pd.read_csv('data/MINDsmall_dev/behaviors.tsv', sep='\t', names=behv_cols, nrows=10000)

news_cols = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
train_news = pd.read_csv('data/MINDsmall_train/news.tsv', sep='\t', names=news_cols, nrows=10000)
dev_news = pd.read_csv('data/MINDsmall_dev/news.tsv', sep='\t', names=news_cols, nrows=10000)

print(f"  Train: {len(train_beh)} impressions, {train_beh['user_id'].nunique()} users")
print(f"  Dev:   {len(dev_beh)} impressions, {dev_beh['user_id'].nunique()} users")
print(f"  Train news: {len(train_news)} articles")
print(f"  Dev news: {len(dev_news)} articles")

print("\nBehaviors sample:")
print(train_beh.head(2))
print("\nNews sample:")
print(train_news.head(2))


Loading MIND Data...
  Train: 10000 impressions, 8683 users
  Dev:   10000 impressions, 9382 users
  Train news: 10000 articles
  Dev news: 10000 articles

Behaviors sample:
   impr_id user_id                   time  \
0        1  U13740  11/11/2019 9:05:58 AM   
1        2  U91836  11/12/2019 6:11:30 PM   

                                                hist  \
0  N55189 N42782 N34694 N45794 N18445 N63302 N104...   
1  N31739 N6072 N63045 N23979 N35656 N43353 N8129...   

                                               imprs  
0                                  N55689-1 N35729-0  
1  N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...  

News sample:
  news_id   category      subcategory  \
0  N55528  lifestyle  lifestyleroyals   
1  N19639     health       weightloss   

                                               title  \
0  The Brands Queen Elizabeth, Prince Charles, an...   
1                      50 Worst Habits For Belly Fat   

                                            abstr

In [5]:
def parse_impressions(impr_str):
    """Parse MIND 'N123-1 N456-0' → list of (news_id, clicked)"""
    items = impr_str.split()
    news_ids = [item.split('-')[0] for item in items]
    clicks = [int(item.split('-')[1]) for item in items]
    return list(zip(news_ids, clicks))

# Test parsing
sample_impr = train_beh['imprs'].iloc[0]
parsed = parse_impressions(sample_impr)

print("Sample impression:", sample_impr[:50]+"...")
print("Parsed clicks:", parsed[:5])
print(f"\nFound {sum(c[1] for c in parsed)} clicks in this impression")

# Count clicks across sample
train_sample = train_beh.head(100)
total_clicks = 0
for impr in train_sample['imprs']:
    clicks = parse_impressions(impr)
    total_clicks += sum(c[1] for c in clicks)

print(f"\nSample stats: {total_clicks} total clicks in 100 impressions")


Sample impression: N55689-1 N35729-0...
Parsed clicks: [('N55689', 1), ('N35729', 0)]

Found 1 clicks in this impression

Sample stats: 137 total clicks in 100 impressions
